# LLM Inference Baseline — Colab GPU run

End-to-end Phase 1: verify GPU, install deps, smoke-test the harness on a tiny model, then run the real long-context sweep, then summarize.

**Before starting:** `Runtime → Change runtime type → T4 GPU` (free) or A100 (Colab Pro+).


## 1. Confirm GPU

In [ ]:
!nvidia-smi

## 2. Get the code

Edit `REPO_URL` to point at your GitHub repo, then run the cell. If the notebook is already inside a clone, the clone is skipped.

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/LLM_Inference.git"  # <-- EDIT THIS

if not os.path.exists("scripts/run_context_sweep.py"):
    !git clone {REPO_URL}
    repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    %cd {repo_name}

!ls

## 3. Install Python deps

Colab already ships PyTorch with CUDA, so we only install the rest.

In [ ]:
!pip install -q -r requirements.txt

## 4. (Optional) Hugging Face auth

Only needed for gated models like Llama-3. Accept the license on the model page first, then create a token at https://huggingface.co/settings/tokens and paste it into the prompt.

In [ ]:
from huggingface_hub import login
login()

## 5. Smoke test (~30s, tiny public model)

Verifies the harness end-to-end before pulling real weights. Should produce two JSONL rows in `results/baseline_hf.jsonl` with non-null TTFT and latency.

In [ ]:
!python scripts/run_context_sweep.py \
  --config configs/baseline_hf.yaml \
  --model-id sshleifer/tiny-gpt2 \
  --dtype float16 \
  --context-lengths 128 512 1024 \
  --max-new-tokens 16

In [ ]:
!cat results/baseline_hf.jsonl

## 6. Real Phase 1 sweep

Edit the cell for your runtime:

- **T4 (16 GB):** use `--dtype float16`. Llama-3-8B weights alone are ~16 GB so loading is tight and OOM at long context is expected — those failure rows are the data point that shows the memory wall.
- **A100 (40 GB):** `bfloat16` is fine for the full 8k→64k sweep.

If you skipped the HF login above, override the model with `--model-id` to something non-gated (e.g. `Qwen/Qwen2.5-1.5B-Instruct`, `microsoft/phi-2`).

In [ ]:
# T4 example (Llama-3-8B in fp16)
!python scripts/run_context_sweep.py \
  --config configs/baseline_hf.yaml \
  --dtype float16 \
  --context-lengths 2048 4096 8192 16384 \
  --max-new-tokens 64

# A100 example — uncomment if applicable
# !python scripts/run_context_sweep.py \
#   --config configs/baseline_hf.yaml \
#   --context-lengths 8192 16384 32768 65536 \
#   --max-new-tokens 128

## 7. Summarize

In [ ]:
!python scripts/summarize_results.py --results-dir results/

## 8. Save results before the runtime disconnects

Free Colab resets `/content` on disconnect. Either download to your laptop or push back to GitHub.

In [ ]:
# Option A: download to laptop
from google.colab import files
files.download("results/baseline_hf.jsonl")

# Option B: push to GitHub from inside Colab (uncomment, edit name/email)
# !git config user.email "you@example.com"
# !git config user.name "Your Name"
# !git add results/ && git commit -m "Colab T4 baseline run" && git push